# 09 — Build a bulk cross-run analysis

This notebook runs the independent post-processing pipeline over a read-only directory tree of completed SpectralBridge flightlines. It creates canonical catalogs, a virtual DuckDB population, balanced synthetic MicaSense-to-Landsat summaries, and leave-one-site-out validation without changing individual NEON or drone runs.

## 1. Configure the source tree and output directory

The default `input_kind="full"` excludes polygon subsets so they are not counted twice when a flightline has both full and polygon merged products. The output and scratch paths must be outside the read-only input tree. Start with `preflight_only = True` to inspect identity, duplicate, and rejection catalogs before expensive analysis.

In [ ]:
from pathlib import Path
from pprint import pprint

import duckdb

from spectralbridge import run_bulk_pipeline

RUN = False
input_root = Path.home() / "processed_spectralbridge"
output_dir = Path.home() / "spectralbridge_bulk_analysis"
temp_directory = Path.home() / "spectralbridge_bulk_scratch"
input_kind = "full"
memory_limit = "8GB"
threads = 4
preflight_only = True
materialize_observations = False

## 2. Build or reuse the collection

The pipeline recovers canonical flightline identity from product filenames rather than outer distributed-compute folders. It rebuilds when the source inventory or scientific settings change and otherwise reports `status="reused"`. After reviewing preflight, set `preflight_only = False` to run translation and held-out-site analyses.

In [ ]:
result = None
if RUN:
    result = run_bulk_pipeline(
        input_root,
        output_dir,
        input_kind=input_kind,
        memory_limit=memory_limit,
        threads=threads,
        temp_directory=temp_directory,
        preflight_only=preflight_only,
        materialize_observations=materialize_observations,
    )
    pprint(result)
else:
    print("Edit the paths and set RUN = True to build the bulk collection.")

## 3. Inspect catalogs and analyses

The database stores canonical flightline and source-file catalogs. Its `bulk_observations` view queries accepted original Parquets directly unless materialization was explicitly enabled.

In [ ]:
if RUN and result is not None:
    with duckdb.connect(result["database"], read_only=True) as con:
        source_summary = con.execute(
            "SELECT status, COUNT(*) AS flightlines, SUM(row_count) AS rows "
            "FROM flightlines GROUP BY status ORDER BY status"
        ).df()
        census = con.execute("SELECT * FROM dataset_census_summary").df()
        coefficients = None
        if not preflight_only:
            coefficients = con.execute(
                "SELECT landsat_sensor, band_index, analysis_level, slope, "
                "intercept, r2, flightline_count, site_count "
                "FROM candidate_translation_coefficients "
                "ORDER BY landsat_sensor, band_index, analysis_level"
            ).df()
    display(source_summary)
    display(census)
    if coefficients is not None:
        display(coefficients)

## 4. Interpret the result

Each equation is `Landsat = slope × MicaSense + intercept`. Pixel-pooled results allow large flightlines to contribute more observations; flightline- and site-balanced results give each replicate equal total weight. These same-source synthetic relationships are separate from fixed percentage brightness adjustment and are not empirical field calibration.

In [ ]:
for artifact in sorted(path for path in output_dir.rglob("*") if path.is_file()):
    print(artifact.relative_to(output_dir), artifact.stat().st_size)